Pulls a handful of COCO images from Hugging Face and lands them in raw.
Photos go straight to RustFS as objects, and a small table describing
them (where each photo is, its boxes/labels) goes into DuckLake.

Standard library imports for byte buffers and JSON encoding.

In [1]:
import io
import json

Imports for S3/RustFS access, DuckDB, and pulling the dataset from Hugging Face.

In [2]:
import boto3
import duckdb
import pandas as pd
from datasets import load_dataset

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration: how many images to pull, and where they land in the RustFS bucket.

In [3]:
N_IMAGES = 60   # keeping this small, see README
BUCKET = "lakehouse"
S3_PREFIX = "assets/coco/images"

Creates a boto3 S3 client pointed at RustFS instead of real AWS.

In [4]:
# connects to RustFS. RustFS speaks the S3 protocol, so boto3 (normally
# used for real AWS) works here too, just pointed at our local endpoint
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url="http://rustfs:9000",
        aws_access_key_id="rustfsadmin",
        aws_secret_access_key="rustfsadmin",
    )

Connects DuckDB and attaches the DuckLake catalog.

In [5]:
# turns on DuckLake, connects it to RustFS, opens/creates the catalog
# with its raw/silver/gold schemas -- same attach script every time
def attach_lakehouse():
    con = duckdb.connect()
    con.execute(open("sql/00_attach.sql").read())
    return con

Uploads a single image to RustFS and builds the metadata row describing it.

In [6]:
# handles one image: uploads it to RustFS, returns a row describing it
def upload_image_and_build_row(s3, index, sample):
    img = sample["image"].convert("RGB")
    objects = sample["objects"]

    # give this image a filename inside the bucket, e.g. "0007.jpg"
    key = f"{S3_PREFIX}/{index:04d}.jpg"

    # turn the PIL image into raw JPEG bytes so we can upload it
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf, ContentType="image/jpeg")

    # bbox/segmentation/categories stay as plain JSON strings here --
    # parsing them into real typed columns is silver's job, not raw's
    return {
        "image_uri": f"s3://{BUCKET}/{key}",
        "width": img.width,
        "height": img.height,
        "bbox_json": json.dumps(objects.get("bbox", [])),
        "segmentation_json": json.dumps(objects.get("segmentation", [])),
        "categories_json": json.dumps(objects.get("categories", [])),
    }

Connects to RustFS and attaches the DuckLake catalog.

In [7]:
s3 = make_s3_client()

con = attach_lakehouse()

Opens a streaming connection to the COCO dataset on Hugging Face.

In [8]:
print(f"Streaming {N_IMAGES} images from ariG23498/coco2017 (validation split)...")

# streaming=True means we don't download the full 20GB dataset,
# just the images we actually ask for
ds = load_dataset("ariG23498/coco2017", split="validation", streaming=True)

Streaming 60 images from ariG23498/coco2017 (validation split)...


Uploads each image to RustFS and collects its metadata row, printing progress every 10 images.

In [9]:
rows = []

for i, sample in enumerate(ds):
    if i >= N_IMAGES:
        break   # got enough, stop early

    rows.append(upload_image_and_build_row(s3, i, sample))

    if (i + 1) % 10 == 0:
        print(f"  ...{i + 1}/{N_IMAGES} images uploaded")

  ...10/60 images uploaded
  ...20/60 images uploaded


  ...30/60 images uploaded
  ...40/60 images uploaded


  ...50/60 images uploaded
  ...60/60 images uploaded


Confirms all images were uploaded to RustFS.

In [10]:
print(f"Uploaded {len(rows)} images to s3://{BUCKET}/{S3_PREFIX}/")

Uploaded 60 images to s3://lakehouse/assets/coco/images/


Writes the collected metadata into `raw.coco_annotations`, creating a new DuckLake snapshot.

In [11]:
# turn our list of dicts into a table and save it into DuckLake --
# this is the moment raw.coco_annotations actually becomes real
df = pd.DataFrame(rows)

con.register("coco_df", df)

con.execute("CREATE OR REPLACE TABLE raw.coco_annotations AS SELECT * FROM coco_df")

Confirms the row count landed in the table.

In [12]:
count = con.sql("SELECT COUNT(*) FROM raw.coco_annotations").fetchone()[0]

print(f"raw.coco_annotations now has {count} rows")

raw.coco_annotations now has 60 rows


Shows the most recent DuckLake snapshots as proof a new version was created.

In [13]:
# show the last few snapshots -- proof this created a new catalog version
print("\nMost recent snapshots:")

con.sql("FROM ducklake_snapshots('lake') ORDER BY snapshot_id DESC LIMIT 5").show()


Most recent snapshots:
┌─────────────┬───────────────────────────────┬────────────────┬───────────────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────────┐
│ snapshot_id │         snapshot_time         │ schema_version │                              changes                              │ author  │ commit_message │ commit_extra_info │
│    int64    │   timestamp with time zone    │     int64      │                      map(varchar, varchar[])                      │ varchar │    varchar     │      varchar      │
├─────────────┼───────────────────────────────┼────────────────┼───────────────────────────────────────────────────────────────────┼─────────┼────────────────┼───────────────────┤
│           4 │ 2026-08-09 05:12:28.462686+00 │              4 │ {tables_created=[raw.coco_annotations], tables_inserted_into=[4]} │ NULL    │ NULL           │ NULL              │
│           3 │ 2026-08-09 05:12:21.563377+00 │              3 │ {schemas_cr